In [1]:
import pandas as pd

# 訓練（train）データとテスト（test）データを読み込む
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# データの行数（データ数）と列数（特徴の種類）を確認
print(f"訓練データのサイズ: {train_df.shape}")
print(f"テストデータのサイズ: {test_df.shape}")

# 訓練データの最初の5行を表示
train_df.head()

訓練データのサイズ: (594194, 21)
テストデータのサイズ: (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# データの読み込み
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# 1. 基本情報の確認
print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

# 2. 欠損値とデータ型の確認
display(train.info())

# 3. 目的変数（Churn）のバランス確認（クラス不均衡のチェック）
print("\nTarget Distribution:")
print(train['Churn'].value_counts(normalize=True))

Train Shape: (594194, 21)
Test Shape: (254655, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract         

None


Target Distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [3]:
import optuna
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# --- 1. データの読み込みと基本前処理 ---
X = train_df.drop(['id', 'Churn'], axis=1)
y = train_df['Churn'].map({'Yes': 1, 'No': 0})
X_test = test_df.drop(['id'], axis=1)

# --- 2. Feature Engineering (FE v1 & v2) ---
# 数値の組み合わせ
X['AvgMonthlyCharges'] = X['TotalCharges'] / (X['tenure'] + 1)
X_test['AvgMonthlyCharges'] = X_test['TotalCharges'] / (X_test['tenure'] + 1)
X['IsNewCustomer'] = (X['tenure'] == 0).astype(int)
X_test['IsNewCustomer'] = (X_test['tenure'] == 0).astype(int)

services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
X['TotalServices'] = (train_df[services] == 'Yes').sum(axis=1)
X_test['TotalServices'] = (test_df[services] == 'Yes').sum(axis=1)

# カテゴリの組み合わせ
X['Contract_Payment'] = train_df['Contract'] + "_" + train_df['PaymentMethod']
X_test['Contract_Payment'] = test_df['Contract'] + "_" + test_df['PaymentMethod']
X['Net_Support'] = train_df['InternetService'] + "_" + train_df['TechSupport']
X_test['Net_Support'] = test_df['InternetService'] + "_" + test_df['TechSupport']

# カテゴリ変数の数値化
cat_features = X.select_dtypes(include=['object']).columns
for col in cat_features:
    le = LabelEncoder()
    full_labels = pd.concat([X[col], X_test[col]]).astype(str)
    le.fit(full_labels)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# --- 3. Optuna によるハイパーパラメータ最適化 ---
def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    
    X_train_sub, X_valid_sub, y_train_sub, y_valid_sub = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    lgb_train = lgb.Dataset(X_train_sub, label=y_train_sub)
    lgb_valid = lgb.Dataset(X_valid_sub, label=y_valid_sub, reference=lgb_train)

    model = lgb.train(
        param, 
        lgb_train, 
        valid_sets=[lgb_valid], 
        valid_names=['valid'],  # ここで名前を 'valid' に固定します
        num_boost_round=1000, 
        callbacks=[lgb.early_stopping(50)]
    )
    
    # これで 'valid' というキーで確実に取得できます
    return model.best_score['valid']['auc']

print("Starting Optuna optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

# --- 4. 最適なパラメータで 5-Fold K-Fold を実行 ---
print(f"\nOptimization finished. Best AUC: {study.best_value}")
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'random_state': 42,
    **study.best_params # Optunaの結果を自動反映
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n=== Fold {fold + 1} / {N_FOLDS} ===")
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)
    
    model = lgb.train(best_params, lgb_train, valid_sets=[lgb_train, lgb_valid],
                      valid_names=['train', 'valid'], num_boost_round=1000,
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    
    oof_preds[valid_idx] = model.predict(X_valid, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

print(f"\nFinal Overall CV AUC with Best Params: {roc_auc_score(y, oof_preds):.5f}")

[I 2026-03-04 19:01:28,296] A new study created in memory with name: no-name-c3f66961-aef3-4c97-8513-dc8b8344b314


Starting Optuna optimization...
Training until validation scores don't improve for 50 rounds


[I 2026-03-04 19:01:49,687] Trial 0 finished with value: 0.9165817930951987 and parameters: {'learning_rate': 0.015569684136584466, 'num_leaves': 87, 'feature_fraction': 0.7457171866966632, 'bagging_fraction': 0.6955582841383591, 'bagging_freq': 7, 'min_child_samples': 20}. Best is trial 0 with value: 0.9165817930951987.


Did not meet early stopping. Best iteration is:
[955]	valid's auc: 0.916582
Training until validation scores don't improve for 50 rounds


[I 2026-03-04 19:01:54,718] Trial 1 finished with value: 0.9165883186630549 and parameters: {'learning_rate': 0.08275090985638721, 'num_leaves': 29, 'feature_fraction': 0.6281512588898593, 'bagging_fraction': 0.47539907174346635, 'bagging_freq': 3, 'min_child_samples': 59}. Best is trial 1 with value: 0.9165883186630549.


...

Early stopping, best iteration is:
[547]	valid's auc: 0.916741
Training until validation scores don't improve for 50 rounds


[I 2026-03-04 19:04:16,238] Trial 18 finished with value: 0.916675926354322 and parameters: {'learning_rate': 0.01232567612851191, 'num_leaves': 67, 'feature_fraction': 0.4866931385715272, 'bagging_fraction': 0.7720139644700592, 'bagging_freq': 7, 'min_child_samples': 44}. Best is trial 16 with value: 0.9168561848943785.


Did not meet early stopping. Best iteration is:
[1000]	valid's auc: 0.916676
Training until validation scores don't improve for 50 rounds


[I 2026-03-04 19:04:29,893] Trial 19 finished with value: 0.9166732230758 and parameters: {'learning_rate': 0.028232058691497926, 'num_leaves': 20, 'feature_fraction': 0.4935037721801353, 'bagging_fraction': 0.764007930623267, 'bagging_freq': 4, 'min_child_samples': 89}. Best is trial 16 with value: 0.9168561848943785.


Did not meet early stopping. Best iteration is:
[1000]	valid's auc: 0.916673

Optimization finished. Best AUC: 0.9168561848943785

=== Fold 1 / 5 ===
Training until validation scores don't improve for 50 rounds
[100]	train's auc: 0.915217	valid's auc: 0.913809
[200]	train's auc: 0.917305	valid's auc: 0.915214
[300]	train's auc: 0.918653	valid's auc: 0.915695
[400]	train's auc: 0.919737	valid's auc: 0.915938
[500]	train's auc: 0.920668	valid's auc: 0.916078
[600]	train's auc: 0.921502	valid's auc: 0.916181
[700]	train's auc: 0.922245	valid's auc: 0.916226
[800]	train's auc: 0.922965	valid's auc: 0.916221
Early stopping, best iteration is:
[769]	train's auc: 0.922738	valid's auc: 0.916237

=== Fold 2 / 5 ===
Training until validation scores don't improve for 50 rounds
[100]	train's auc: 0.914962	valid's auc: 0.914893
[200]	train's auc: 0.917102	valid's auc: 0.916238
[300]	train's auc: 0.918439	valid's auc: 0.91672
[400]	train's auc: 0.919499	valid's auc: 0.916917
[500]	train's auc: 0.920

In [4]:
# 最適化済みモデルによる提出ファイルの作成
submission_optuna = pd.read_csv("data/sample_submission.csv")
submission_optuna['Churn'] = test_preds # 5-Fold平均（最適パラメータ版）
submission_optuna.to_csv("submission_optuna.csv", index=False)

print("submission_optuna.csv has been created!")

submission_optuna.csv has been created!
